# TAB 63: exp63_bidirectional_energy (exp62 base + bi_directional_relative_energy) - Colab
# H15: Bi-directional Spatial Energy Contrast (signed Delta E + absolute magnitude |Delta E|)
# Eliminates Rivet_v2 half-space PC1 sign inversion by mapping deficit hole flux and excess scattering flux into positive anomaly subspace.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, glob
DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)
# Verify: TDMS + GT masks phai thay tu Drive
tdms = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print('TDMS:', len(tdms))
for p in [os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')]:
    print(('OK  ' if os.path.exists(p) else 'MISS') + ' ' + p)

In [ ]:
!test -d GraduationProject && (cd GraduationProject && git pull --ff-only) || git clone https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git switch Define_PECT_JEPA && git pull --ff-only && git log --oneline -3

In [ ]:
# [FIX] sklearn version co sample_weight trong MLPClassifier.fit -> ep dung 1.7.2 (giong env lab)
!pip install -q -U "scikit-learn==1.7.2" nptdms scipy tqdm 2>&1 | tail -2
!cd GraduationProject && python -c "import torch, sklearn, scipy, nptdms; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| sklearn:', sklearn.__version__)"


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
# ===== CONFIG EXP63 (H15) =====
EXP_NAME = 'exp63_bidirectional_energy'
BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement true --carrier_flat_weight 0.0 --scattering_var_weight 0.5 --inv_start_epoch 11 --inv_ramp_epochs 20 --xcov_start_epoch 11 --xcov_ramp_epochs 20 --inv_cosine true --zc_cond_weight 0.1 --var_gamma 0.5 --offset_weight 0.0 --pool_var_weight 0.0 --device cuda'
EXTRA = '--continuous_pos_embed true --contiguous_masking true --predictor_scattering_only false --scan_centered_inv true --inv_predictor true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --scattering_cov_weight 0.2 --inv_weight 0.2 --xcov_weight 0.05'
os.environ.update({'EXP_NAME': EXP_NAME, 'BASE_FLAGS': BASE_FLAGS, 'EXTRA': EXTRA, 'DATA': DATA, 'DRIVE_RUNS': DRIVE_RUNS})
print(EXP_NAME)
print(EXTRA)

In [ ]:
# Symlink runs sang Drive: checkpoint va metrics ghi truc tiep len Google Drive
!mkdir -p "$DRIVE_RUNS"
!if [ -e GraduationProject/experiments/cross_condition/runs ] && [ ! -L GraduationProject/experiments/cross_condition/runs ]; then mv GraduationProject/experiments/cross_condition/runs GraduationProject/experiments/cross_condition/runs_local_bak; fi
!ln -sfn "$DRIVE_RUNS" GraduationProject/experiments/cross_condition/runs
!ls -la GraduationProject/experiments/cross_condition/ | grep runs


In [ ]:
# Cell Train (Fresh Run): chay de huan luyen exp63 (H15)
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME --add_timestamp true $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1

In [ ]:
# ==============================================================================
# Full Downstream Evaluation + Readouts Diagnostic sau khi train xong
# ==============================================================================
import os, glob

run_dir = os.environ.get('LAST_RUN_DIR', None)
if not run_dir or not os.path.exists(run_dir):
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")))
    if not candidates:
        raise FileNotFoundError(f"Khong tim thay thu muc run nao cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")

eval_ckpt = ckpt_best if os.path.exists(ckpt_best) else ckpt_latest
if not os.path.exists(eval_ckpt):
    raise FileNotFoundError(f"Khong tim thay checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Dang evaluate checkpoint: {eval_ckpt}")
os.environ["EVAL_CKPT"] = eval_ckpt

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.evaluate --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.diagnose_readouts --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda